# BharatLipi — Module 3 Backend + Module 1/2 OCR + IndicTrans2 Translation

This notebook runs the real API used by the existing React website. It preserves the existing Google Drive SQLite repository and the 1,199 manuscript image pairs.

**Upload workflow:** image(s), multi-page PDF, or ZIP → Module 1 image preprocessing → Module 2 Tesseract OCR + Sanskrit text normalization → optional IndicTrans2 translation into selected language(s) → saved manuscript + translation records.

**Translation:** each manuscript can store several language versions. Hindi and English are default upload targets; the user can select from all configured IndicTrans2 language/script options. The original OCR text is never replaced by a translation.

**Accuracy note:** machine translation cannot guarantee perfect word-for-word fidelity for historical Sanskrit or uncertain OCR. The website retains the OCR source and flags machine-generated translations for human scholarly review.

Run the cells in order from a fresh Colab runtime. Accept access to all three model repositories and authenticate once with a Hugging Face read token. The notebook verifies package imports, checks access to each model repository, and runs real Sanskrit-to-English and Sanskrit-to-Hindi translation smoke tests before starting the API. A GPU runtime is strongly recommended.


In [ ]:
# 1. Install API, OCR, and IndicTrans2 dependencies
# Run this cell first in a fresh Colab runtime.

import sys
import subprocess
import importlib
import importlib.metadata as metadata

packages = [
    "fastapi>=0.110,<1",
    "uvicorn[standard]>=0.27,<1",
    "pyngrok>=7,<8",
    "requests",
    "python-multipart",
    "pytesseract",
    "opencv-python-headless",
    "pillow",
    "pymupdf",
    "numpy>=2.1,<2.5",
    "transformers>=4.51,<5",
    "accelerate>=0.33,<2",
    "sentencepiece",
    "sacremoses",
    "sacrebleu",
    "indic-nlp-library-itt",
    "indictranstoolkit==1.1.1",
]

print("Installing BharatLipi dependencies into:", sys.executable)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "--upgrade-strategy", "only-if-needed", *packages
])

# Explicitly reinstall the toolkit wheel so a prior partial/failed install cannot
# leave the current kernel without the importable package.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "--force-reinstall", "--no-deps", "indictranstoolkit==1.1.1"
])

importlib.invalidate_caches()

import torch
import numpy as np
import transformers
import huggingface_hub
from packaging.version import Version

if Version(torch.__version__.split("+")[0]) < Version("2.5"):
    raise RuntimeError(
        f"PyTorch {torch.__version__} detected. IndicTransToolkit recommends PyTorch >=2.5. "
        "Select a current Colab runtime, then rerun this cell."
    )

if Version(np.__version__) < Version("2.1"):
    raise RuntimeError(f"NumPy {np.__version__} is too old; NumPy >=2.1 is required.")

# Import path recommended by the toolkit maintainers, with documented fallbacks.
try:
    from IndicTransToolkit import IndicProcessor
    INDIC_PROCESSOR_IMPORT_PATH = "IndicTransToolkit"
except ImportError:
    try:
        from IndicTransToolkit.IndicTransToolkit import IndicProcessor
        INDIC_PROCESSOR_IMPORT_PATH = "IndicTransToolkit.IndicTransToolkit"
    except ImportError:
        from IndicTransToolkit.processor import IndicProcessor
        INDIC_PROCESSOR_IMPORT_PATH = "IndicTransToolkit.processor"

print("\n========== DEPENDENCY CHECK ==========")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("IndicTransToolkit:", metadata.version("indictranstoolkit"))
print("IndicProcessor import:", INDIC_PROCESSOR_IMPORT_PATH)
print("Dependency and IndicProcessor import checks passed.")
print("=====================================")


In [ ]:
# 1A. Install the same Tesseract language/script data used by Module 2
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-san tesseract-ocr-script-deva

import subprocess
language_listing = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True, check=True).stdout
available_tesseract_languages = set(language_listing.splitlines()[1:])
required_tesseract_languages = {"san", "Devanagari"}
missing_tesseract_languages = required_tesseract_languages - available_tesseract_languages
print("Tesseract languages available:", sorted(available_tesseract_languages))
if missing_tesseract_languages:
    raise RuntimeError("Missing Tesseract language data: " + ", ".join(sorted(missing_tesseract_languages)))
print("Sanskrit + Devanagari OCR models are ready.")


In [ ]:
# 2. Mount Google Drive and verify the existing Module 3 repository
from google.colab import drive
from pathlib import Path
import os
import sqlite3

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/BharatLipi")
MODULE3_DIR = BASE_DIR / "Module3"
DB_PATH = MODULE3_DIR / "database" / "bharatlipi.db"
DATASET_DIR = MODULE3_DIR / "dataset" / "preprocessed"
ENHANCED_DIR = DATASET_DIR / "Enhanced_Grayscale"
BINARY_DIR = DATASET_DIR / "Binary_Images"
UPLOADS_DIR = MODULE3_DIR / "uploads"
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR = MODULE3_DIR / "models" / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(MODEL_CACHE_DIR / "hub")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("Database:", DB_PATH)
print("Dataset:", DATASET_DIR)
print("New upload storage:", UPLOADS_DIR)
print("Persistent translation model cache:", MODEL_CACHE_DIR)

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the existing database at {DB_PATH}. "
        "Open your original Module3 notebook and run its database/import cells first."
    )

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the extracted dataset at {DATASET_DIR}. "
        "Expected the preprocessed dataset folder in Google Drive."
    )

if not ENHANCED_DIR.exists() or not BINARY_DIR.exists():
    raise FileNotFoundError(
        "The Enhanced_Grayscale and/or Binary_Images directory is missing. "
        "Check the extracted dataset folder names."
    )

REQUIRED_COLUMNS = {
    "id", "manuscript_id", "title", "original_filename", "original_path",
    "enhanced_path", "binary_path", "raw_ocr_text", "cleaned_text",
    "translated_text", "verified_text", "language", "script", "category",
    "description", "source", "processing_status", "verification_status",
    "created_at", "updated_at"
}

with sqlite3.connect(str(DB_PATH), timeout=30) as conn:
    conn.row_factory = sqlite3.Row
    table_exists = conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name='manuscripts'"
    ).fetchone()
    if not table_exists:
        raise RuntimeError("The database exists, but its 'manuscripts' table is missing.")

    actual_columns = {
        row["name"] for row in conn.execute("PRAGMA table_info(manuscripts)").fetchall()
    }
    missing_columns = sorted(REQUIRED_COLUMNS - actual_columns)
    if missing_columns:
        raise RuntimeError(
            "Your manuscripts table is missing expected columns: " + ", ".join(missing_columns)
        )

    total_records = conn.execute("SELECT COUNT(*) FROM manuscripts").fetchone()[0]
    paired_records = conn.execute("""
        SELECT COUNT(*) FROM manuscripts
        WHERE enhanced_path IS NOT NULL AND binary_path IS NOT NULL
    """).fetchone()[0]
    first_record = conn.execute("""
        SELECT manuscript_id, original_filename, enhanced_path, binary_path
        FROM manuscripts ORDER BY manuscript_id LIMIT 1
    """).fetchone()


def count_image_files(folder: Path) -> int:
    valid = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in valid)

print("\n========== REPOSITORY CHECK ==========")
print("Database records:", total_records)
print("Records with both image paths:", paired_records)
print("Enhanced image files:", count_image_files(ENHANCED_DIR))
print("Binary image files:", count_image_files(BINARY_DIR))
print("Sample record:", dict(first_record) if first_record else "No records")
print("======================================")

if total_records == 0:
    print("WARNING: The database is empty. The API will start, but the website will show no manuscripts.")
if paired_records != total_records:
    print("WARNING: Some records do not have both enhanced and binary paths; those image panels may be unavailable.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Database: /content/drive/MyDrive/BharatLipi/Module3/database/bharatlipi.db
Dataset: /content/drive/MyDrive/BharatLipi/Module3/dataset/preprocessed
New upload storage: /content/drive/MyDrive/BharatLipi/Module3/uploads
Persistent translation model cache: /content/drive/MyDrive/BharatLipi/Module3/models/huggingface_cache

========== REPOSITORY CHECK ==========
Database records: 1208
Records with both image paths: 1208
Enhanced image files: 1199
Binary image files: 1199
Sample record: {'manuscript_id': 'BL-0001', 'original_filename': 'image_1.png', 'enhanced_path': '/content/drive/MyDrive/BharatLipi/Module3/dataset/preprocessed/Enhanced_Grayscale/image_1.png', 'binary_path': '/content/drive/MyDrive/BharatLipi/Module3/dataset/preprocessed/Binary_Images/image_1.png'}


In [ ]:
# 2A. Authenticate once and verify access to every IndicTrans2 checkpoint
# IMPORTANT: request/accept access to all three model pages using the SAME HF account.
# The token is entered through a hidden prompt and is not written into this notebook or Google Drive.

import os
from pathlib import Path
from getpass import getpass
from huggingface_hub import login, whoami, HfApi, hf_hub_download

HF_TOKEN = getpass("Enter your Hugging Face READ token (input is hidden): ").strip()
if not HF_TOKEN:
    raise ValueError("Hugging Face token is empty. Create a READ token and run this cell again.")

# Keep token available to FastAPI background tasks and Transformers in this runtime.
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

account = whoami(token=HF_TOKEN)
print("Authenticated Hugging Face account:", account.get("name", "verified account"))

MODEL_REPOSITORIES = {
    "Indic to English": "ai4bharat/indictrans2-indic-en-dist-200M",
    "English to Indic": "ai4bharat/indictrans2-en-indic-dist-200M",
    "Indic to Indic": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

api = HfApi(token=HF_TOKEN)
access_failures = []

for label, repo_id in MODEL_REPOSITORIES.items():
    print(f"\nChecking {label}: {repo_id}")
    try:
        repo_files = api.list_repo_files(repo_id=repo_id, token=HF_TOKEN)
        if "config.json" not in repo_files:
            raise RuntimeError("Repository is visible, but config.json was not found.")

        # Download the small config and remote Python configuration/model files now.
        # This catches gated-code-file failures before the API is launched.
        files_to_check = ["config.json"]
        files_to_check.extend(
            name for name in repo_files
            if name.endswith(".py")
            and Path(name).name.startswith(("configuration_", "modeling_"))
        )
        for filename in dict.fromkeys(files_to_check):
            hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                token=HF_TOKEN,
                cache_dir=str(MODEL_CACHE_DIR),
            )
        print(f"ACCESS OK — {len(repo_files)} repository files visible; config/code downloads passed.")

    except Exception as exc:
        message = f"{label} ({repo_id}): {type(exc).__name__}: {exc}"
        access_failures.append(message)
        print("ACCESS CHECK FAILED:", message)

if access_failures:
    raise RuntimeError(
        "Hugging Face access is not ready for every required model. "
        "Open each model page in a browser, sign in to the SAME account shown above, "
        "accept/request access, wait for approval if required, then rerun this cell.\n\n"
        + "\n".join(access_failures)
    )

print("\nAll three IndicTrans2 repositories are accessible to this account.")
print("Model weights will download to the persistent Google Drive cache on first use.")


## 3A. Enable IndicTrans2 model access

Translation uses three IndicTrans2 distilled checkpoints. Open each page below, sign in to the SAME Hugging Face account, and accept/request access. Some gated repositories require author approval; a token cannot bypass that requirement.

- [Indic → English (200M)](https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M)
- [Indic → Indic (320M)](https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M)
- [English → Indic (200M)](https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M)

Create a **read** token at [Hugging Face Access Tokens](https://huggingface.co/settings/tokens). Enter it in the hidden prompt in the authentication/access-check cell immediately above. Never paste the token into notebook source or share it in chat. The notebook tests access to every repository and its remote Python config files before starting FastAPI.

Use a Colab GPU runtime (Runtime → Change runtime type → T4 GPU) for practical translation speed. CPU mode is supported but can be significantly slower. Model weights are cached under the BharatLipi Module3 folder in Google Drive after first download.


In [ ]:
# 2B. Confirm Hugging Face authentication is available to the API

from huggingface_hub import get_token

_runtime_hf_token = os.environ.get("HF_TOKEN", "").strip() or (get_token() or "").strip()
if not _runtime_hf_token:
    raise RuntimeError("Hugging Face token is missing. Run the authentication/access-check cell above.")

# Keep the token in the environment for model loading and FastAPI background tasks.
os.environ["HF_TOKEN"] = _runtime_hf_token
del _runtime_hf_token
print("Hugging Face authentication is available to the translation backend.")


In [ ]:
# 3. Build the FastAPI application

from typing import Any, Optional
import os, re, io, uuid, copy, shutil, zipfile, threading, unicodedata, subprocess, hashlib
from datetime import datetime, timezone
from pathlib import Path
from contextlib import contextmanager
import sqlite3

from fastapi import FastAPI, HTTPException, Query, Body, UploadFile, File, Form, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

APP_TITLE = "BharatLipi Module 3 API"
API_VERSION = "1.2.0"

app = FastAPI(
    title=APP_TITLE,
    version=API_VERSION,
    description="Knowledge Repository, Discovery, and Sanskrit/Devanagari OCR API for BharatLipi.",
)

# The React/Vite development server and ngrok URL have different origins.
# Wildcard CORS is acceptable for this temporary demonstration API only.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST", "PUT", "PATCH", "OPTIONS"],
    allow_headers=["*"],
)

DB_PATH = Path(DB_PATH)
DATASET_DIR = Path(DATASET_DIR).resolve()
UPLOADS_DIR = Path(UPLOADS_DIR).resolve()
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)


@contextmanager
def db_connection():
    """Open and always close a short-lived SQLite connection per request."""
    if not DB_PATH.exists():
        raise RuntimeError(f"Database file is missing: {DB_PATH}")
    conn = sqlite3.connect(str(DB_PATH), timeout=30)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA busy_timeout = 30000")
    try:
        yield conn
    finally:
        conn.close()


def ensure_translation_schema():
    """Additive schema migration; never alters or deletes existing manuscript records."""
    with db_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS manuscript_translations (
                manuscript_id TEXT NOT NULL,
                language_code TEXT NOT NULL,
                language_name TEXT NOT NULL,
                source_language TEXT NOT NULL,
                source_hash TEXT NOT NULL,
                source_text TEXT NOT NULL,
                translated_text TEXT NOT NULL,
                engine TEXT NOT NULL DEFAULT 'IndicTrans2',
                review_status TEXT NOT NULL DEFAULT 'machine_generated',
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                PRIMARY KEY (manuscript_id, language_code)
            )
        """)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_manuscript_translations_id ON manuscript_translations(manuscript_id)")
        conn.commit()


ensure_translation_schema()


def clean_text(value: Any) -> str:
    return "" if value is None else str(value).strip()


def value_or(value: Any, fallback: str) -> str:
    value = clean_text(value)
    return value if value else fallback


MANUSCRIPT_SELECT = """
    SELECT manuscripts.* ,
           EXISTS(SELECT 1 FROM manuscript_translations mt
                  WHERE mt.manuscript_id = manuscripts.manuscript_id) AS has_saved_translations,
           COALESCE((SELECT GROUP_CONCAT(language_code, '|') FROM manuscript_translations mt
                     WHERE mt.manuscript_id = manuscripts.manuscript_id), '') AS saved_translation_codes
    FROM manuscripts
"""


def derived_status(row: sqlite3.Row) -> str:
    """Translate the existing database statuses into the website's status vocabulary."""
    verification = clean_text(row["verification_status"]).lower()
    processing = clean_text(row["processing_status"]).lower()
    row_keys = set(row.keys())
    has_translation = bool(clean_text(row["translated_text"])) or ("has_saved_translations" in row_keys and bool(row["has_saved_translations"]))
    has_ocr = any(clean_text(row[k]) for k in ("verified_text", "cleaned_text", "raw_ocr_text"))

    if verification == "verified":
        return "ocr-verified"
    if has_translation:
        return "translation-available"
    if has_ocr:
        return "needs-review"
    if processing in {"preprocessed", "digitized", "archived"}:
        return "digitized"
    return "processing"


def public_record(row: sqlite3.Row) -> dict[str, Any]:
    manuscript_id = clean_text(row["manuscript_id"])
    enhanced_path = clean_text(row["enhanced_path"])
    binary_path = clean_text(row["binary_path"])
    image_base = f"/api/manuscripts/{manuscript_id}/images"
    edited_text = (
        clean_text(row["verified_text"])
        or clean_text(row["cleaned_text"])
        or clean_text(row["raw_ocr_text"])
    )
    category = clean_text(row["category"])
    row_keys = set(row.keys())
    saved_translation_codes = clean_text(row["saved_translation_codes"]) if "saved_translation_codes" in row_keys else ""
    saved_translation_codes = [code for code in saved_translation_codes.split("|") if code]
    has_saved_translations = bool(row["has_saved_translations"]) if "has_saved_translations" in row_keys else False

    return {
        # Fields aligned to the existing React data.ts Manuscript interface.
        "id": manuscript_id,
        "manuscriptId": manuscript_id,
        "title": value_or(row["title"], f"Manuscript {manuscript_id}"),
        "script": value_or(row["script"], "Unknown"),
        "language": value_or(row["language"], "Unknown"),
        "period": None,
        "region": None,
        "subjects": [category] if category and category.lower() != "unclassified" else [],
        "keywords": [],
        "status": derived_status(row),
        "ocrConfidence": None,
        "translationConfidence": None,
        "description": value_or(
            row["description"],
            "Descriptive metadata has not yet been catalogued for this manuscript."
        ),
        "folios": None,
        "source": clean_text(row["source"]) or None,
        "lastUpdated": clean_text(row["updated_at"] or row["created_at"]) or None,
        "thumbnail": f"{image_base}/enhanced" if enhanced_path else "",
        "ocrText": edited_text,
        "translation": clean_text(row["translated_text"]),
        # Additional fields needed by the viewer/editor and for transparent metadata.
        "originalFilename": clean_text(row["original_filename"]),
        "processingStatus": value_or(row["processing_status"], "Unknown"),
        "verificationStatus": value_or(row["verification_status"], "Pending"),
        "rawOcrText": clean_text(row["raw_ocr_text"]),
        "cleanedText": clean_text(row["cleaned_text"]),
        "verifiedText": clean_text(row["verified_text"]),
        "translatedText": clean_text(row["translated_text"]),
        "hasTranslations": bool(clean_text(row["translated_text"])) or has_saved_translations,
        "translationLanguages": saved_translation_codes,
        "imageUrls": {
            "enhanced": f"{image_base}/enhanced" if enhanced_path else None,
            "binary": f"{image_base}/binary" if binary_path else None,
        },
    }


def manuscript_row(manuscript_id: str) -> Optional[sqlite3.Row]:
    with db_connection() as conn:
        return conn.execute(
            MANUSCRIPT_SELECT + " WHERE manuscripts.manuscript_id = ?",
            (manuscript_id,),
        ).fetchone()


def dashboard_payload() -> dict[str, Any]:
    with db_connection() as conn:
        counts = conn.execute("""
            SELECT
                COUNT(*) AS total,
                SUM(CASE WHEN LOWER(COALESCE(processing_status, '')) NOT IN ('uploaded', '') THEN 1 ELSE 0 END) AS processed,
                SUM(CASE WHEN COALESCE(TRIM(translated_text), '') <> '' OR EXISTS (SELECT 1 FROM manuscript_translations mt WHERE mt.manuscript_id = manuscripts.manuscript_id) THEN 1 ELSE 0 END) AS translated,
                SUM(CASE WHEN COALESCE(TRIM(raw_ocr_text), '') <> ''
                          OR COALESCE(TRIM(cleaned_text), '') <> ''
                          OR COALESCE(TRIM(verified_text), '') <> '' THEN 1 ELSE 0 END) AS searchable,
                SUM(CASE WHEN LOWER(COALESCE(verification_status, '')) = 'verified' THEN 1 ELSE 0 END) AS verified,
                SUM(CASE WHEN LOWER(COALESCE(verification_status, '')) = 'pending' THEN 1 ELSE 0 END) AS pending_verification,
                SUM(CASE WHEN LOWER(COALESCE(verification_status, '')) = 'pending'
                          AND (COALESCE(TRIM(raw_ocr_text), '') <> ''
                            OR COALESCE(TRIM(cleaned_text), '') <> ''
                            OR COALESCE(TRIM(verified_text), '') <> '') THEN 1 ELSE 0 END) AS pending_review
            FROM manuscripts
        """).fetchone()
        recent_rows = conn.execute(MANUSCRIPT_SELECT + """
            ORDER BY COALESCE(manuscripts.updated_at, manuscripts.created_at) DESC, manuscripts.manuscript_id ASC
            LIMIT 5
        """).fetchall()

    return {
        "total": int(counts["total"] or 0),
        "processed": int(counts["processed"] or 0),
        "translated": int(counts["translated"] or 0),
        "searchable": int(counts["searchable"] or 0),
        "verified": int(counts["verified"] or 0),
        "pendingVerification": int(counts["pending_verification"] or 0),
        "pendingReview": int(counts["pending_review"] or 0),
        "recent": [public_record(row) for row in recent_rows],
    }


def append_status_filter(where: list[str], params: list[Any], status: Optional[str]):
    if not status or status.strip().lower() in {"all", "all statuses"}:
        return
    normalized = status.strip().lower().replace(" ", "-").replace("_", "-")
    if normalized in {"ocr-verified", "verified"}:
        where.append("LOWER(COALESCE(verification_status, '')) = 'verified'")
    elif normalized in {"translation-available", "translated"}:
        where.append("(COALESCE(TRIM(translated_text), '') <> '' OR EXISTS (SELECT 1 FROM manuscript_translations mt WHERE mt.manuscript_id = manuscripts.manuscript_id))")
    elif normalized in {"needs-review", "pending-review"}:
        where.append("LOWER(COALESCE(verification_status, '')) = 'pending'")
        where.append("(COALESCE(TRIM(raw_ocr_text), '') <> '' OR COALESCE(TRIM(cleaned_text), '') <> '' OR COALESCE(TRIM(verified_text), '') <> '')")
    elif normalized in {"digitized", "preprocessed"}:
        where.append("LOWER(COALESCE(processing_status, '')) IN ('preprocessed', 'digitized')")
    elif normalized in {"processing", "uploaded"}:
        where.append("LOWER(COALESCE(processing_status, '')) IN ('uploaded', 'processing')")
    elif normalized in {"pending", "rejected"}:
        where.append("LOWER(COALESCE(verification_status, '')) = ?")
        params.append(normalized)
    else:
        # Also accept the literal processing-status labels stored in SQLite.
        where.append("LOWER(COALESCE(processing_status, '')) = ?")
        params.append(status.strip().lower())


@app.get("/")
def root():
    return {
        "name": APP_TITLE,
        "version": API_VERSION,
        "docs": "/docs",
        "health": "/api/health",
    }


@app.get("/api/health")
def health():
    try:
        with db_connection() as conn:
            count = conn.execute("SELECT COUNT(*) FROM manuscripts").fetchone()[0]
        return {
            "ok": True,
            "databaseConnected": True,
            "manuscriptCount": int(count),
            "datasetDirectoryExists": DATASET_DIR.exists(),
            "uploadDirectoryExists": UPLOADS_DIR.exists(),
            "ocrReady": {"san", "Devanagari"}.issubset(_available_tesseract_models()),
            "translationDependenciesReady": not bool(_TRANSLATION_IMPORT_ERROR),
            "translationModelAccessTokenConfigured": bool(os.environ.get("HF_TOKEN")),
            "translationLanguages": len(SUPPORTED_TRANSLATION_LANGUAGES),
        }
    except Exception as exc:
        raise HTTPException(status_code=503, detail=f"Backend check failed: {exc}")


@app.get("/api/dashboard")
@app.get("/api/stats")
def get_dashboard():
    return dashboard_payload()


@app.get("/api/filters")
def get_filter_options():
    with db_connection() as conn:
        languages = [r[0] for r in conn.execute(
            "SELECT DISTINCT language FROM manuscripts WHERE COALESCE(TRIM(language), '') <> '' ORDER BY language"
        ).fetchall()]
        scripts = [r[0] for r in conn.execute(
            "SELECT DISTINCT script FROM manuscripts WHERE COALESCE(TRIM(script), '') <> '' ORDER BY script"
        ).fetchall()]
        categories = [r[0] for r in conn.execute(
            "SELECT DISTINCT category FROM manuscripts WHERE COALESCE(TRIM(category), '') <> '' ORDER BY category"
        ).fetchall()]
    return {"languages": languages, "scripts": scripts, "categories": categories}


@app.get("/api/manuscripts")
def list_manuscripts(
    q: Optional[str] = Query(default=None, description="Search ID, title, filename, or extracted text"),
    query: Optional[str] = Query(default=None, description="Alias for q"),
    status: Optional[str] = None,
    script: Optional[str] = None,
    language: Optional[str] = None,
    category: Optional[str] = None,
    page: int = Query(default=1, ge=1),
    page_size: int = Query(default=24, ge=1, le=100),
    sort_by: str = Query(default="manuscript_id"),
    sort_order: str = Query(default="asc"),
):
    search_value = (q or query or "").strip()
    where = ["1=1"]
    params: list[Any] = []

    if search_value:
        wildcard = f"%{search_value}%"
        where.append("""(
            manuscript_id LIKE ? OR title LIKE ? OR original_filename LIKE ?
            OR raw_ocr_text LIKE ? OR cleaned_text LIKE ? OR verified_text LIKE ?
            OR translated_text LIKE ?
            OR EXISTS (SELECT 1 FROM manuscript_translations mt
                       WHERE mt.manuscript_id = manuscripts.manuscript_id
                         AND mt.translated_text LIKE ?)
        )""")
        params.extend([wildcard] * 8)
    if script and script.lower() not in {"all", "all scripts"}:
        where.append("LOWER(COALESCE(script, '')) = LOWER(?)")
        params.append(script)
    if language and language.lower() not in {"all", "all languages"}:
        where.append("LOWER(COALESCE(language, '')) = LOWER(?)")
        params.append(language)
    if category and category.lower() not in {"all", "all subjects"}:
        where.append("LOWER(COALESCE(category, '')) = LOWER(?)")
        params.append(category)
    append_status_filter(where, params, status)

    sort_columns = {
        "id": "manuscript_id",
        "manuscript_id": "manuscript_id",
        "title": "title",
        "script": "script",
        "updated": "updated_at",
        "updated_at": "updated_at",
    }
    sort_column = sort_columns.get(sort_by.lower(), "manuscript_id")
    direction = "DESC" if sort_order.lower() == "desc" else "ASC"
    where_sql = " AND ".join(where)
    offset = (page - 1) * page_size

    with db_connection() as conn:
        total = conn.execute(
            f"SELECT COUNT(*) FROM manuscripts WHERE {where_sql}", params
        ).fetchone()[0]
        rows = conn.execute(
            MANUSCRIPT_SELECT + f" WHERE {where_sql} ORDER BY {sort_column} {direction} LIMIT ? OFFSET ?",
            [*params, page_size, offset],
        ).fetchall()

    return {
        "items": [public_record(row) for row in rows],
        "total": int(total),
        "page": page,
        "pageSize": page_size,
        "totalPages": (int(total) + page_size - 1) // page_size,
    }


@app.get("/api/search")
def search_manuscripts(q: str = Query(min_length=1), page: int = Query(default=1, ge=1), page_size: int = Query(default=24, ge=1, le=100)):
    # Pass every argument explicitly so FastAPI's Query defaults are not treated as values.
    return list_manuscripts(
        q=q, query=None, status=None, script=None, language=None, category=None,
        page=page, page_size=page_size, sort_by="manuscript_id", sort_order="asc"
    )


@app.get("/api/manuscripts/{manuscript_id}")
def get_manuscript(manuscript_id: str):
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found")
    return public_record(row)


@app.get("/api/manuscripts/{manuscript_id}/text")
def get_manuscript_text(manuscript_id: str):
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found")
    return {
        "manuscriptId": manuscript_id,
        "rawOcrText": clean_text(row["raw_ocr_text"]),
        "cleanedText": clean_text(row["cleaned_text"]),
        "verifiedText": clean_text(row["verified_text"]),
        "translatedText": clean_text(row["translated_text"]),
    }


@app.get("/api/manuscripts/{manuscript_id}/images/{image_type}")
def get_manuscript_image(manuscript_id: str, image_type: str):
    if image_type not in {"enhanced", "binary"}:
        raise HTTPException(status_code=404, detail="Image type must be 'enhanced' or 'binary'")
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found")

    path_column = "enhanced_path" if image_type == "enhanced" else "binary_path"
    stored_path = clean_text(row[path_column])
    if not stored_path:
        raise HTTPException(status_code=404, detail=f"{image_type.title()} image path is not recorded")

    candidate = Path(stored_path).resolve()
    approved_roots = (DATASET_DIR, UPLOADS_DIR)
    if not any(_is_within(candidate, root) for root in approved_roots):
        raise HTTPException(status_code=403, detail="Stored image path is outside approved dataset/upload storage")

    if not candidate.is_file():
        raise HTTPException(status_code=404, detail=f"Image file does not exist: {candidate.name}")

    allowed_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
    if candidate.suffix.lower() not in allowed_extensions:
        raise HTTPException(status_code=415, detail="Unsupported image format")

    return FileResponse(candidate)


@app.put("/api/manuscripts/{manuscript_id}/text")
def save_manuscript_text(manuscript_id: str, payload: dict[str, Any] = Body(...)):
    """Save OCR correction and/or translation. Accepts camelCase and snake_case keys."""
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found")

    key_map = {
        "cleaned_text": ("cleanedText", "cleaned_text"),
        "raw_ocr_text": ("rawOcrText", "raw_ocr_text"),
        "translated_text": ("translatedText", "translated_text"),
    }
    assignments = []
    values: list[Any] = []
    for column, aliases in key_map.items():
        for key in aliases:
            if key in payload:
                value = payload[key]
                if value is not None and not isinstance(value, str):
                    raise HTTPException(status_code=422, detail=f"{key} must be a string or null")
                assignments.append(f"{column} = ?")
                values.append(value)
                break

    if not assignments:
        raise HTTPException(status_code=422, detail="Provide cleanedText, rawOcrText, or translatedText")

    if any(key in payload for key in ("cleanedText", "cleaned_text", "rawOcrText", "raw_ocr_text")):
        assignments.append("processing_status = 'Text Edited'")
    assignments.append("updated_at = CURRENT_TIMESTAMP")
    values.append(manuscript_id)

    with db_connection() as conn:
        conn.execute(
            f"UPDATE manuscripts SET {', '.join(assignments)} WHERE manuscript_id = ?",
            values,
        )
        conn.commit()

    updated = manuscript_row(manuscript_id)
    return {"ok": True, "message": "Manuscript text saved", "manuscript": public_record(updated)}


@app.put("/api/manuscripts/{manuscript_id}/verification")
def update_verification(manuscript_id: str, payload: dict[str, Any] = Body(...)):
    """Persist Pending / Verified / Rejected status. Verified records must have text."""
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found")

    status = str(payload.get("status", "Verified")).strip().title()
    if status not in {"Pending", "Verified", "Rejected"}:
        raise HTTPException(status_code=422, detail="status must be Pending, Verified, or Rejected")

    supplied_text = payload.get("verifiedText", payload.get("verified_text"))
    if supplied_text is not None and not isinstance(supplied_text, str):
        raise HTTPException(status_code=422, detail="verifiedText must be a string or null")

    if status == "Verified":
        verified_text = (
            supplied_text
            if supplied_text is not None
            else clean_text(row["cleaned_text"]) or clean_text(row["raw_ocr_text"])
        )
        if not verified_text.strip():
            raise HTTPException(
                status_code=409,
                detail="This manuscript has no OCR text to verify yet. Save corrected text first."
            )
        with db_connection() as conn:
            conn.execute("""
                UPDATE manuscripts
                SET verified_text = ?, verification_status = 'Verified',
                    updated_at = CURRENT_TIMESTAMP
                WHERE manuscript_id = ?
            """, (verified_text, manuscript_id))
            conn.commit()
    else:
        with db_connection() as conn:
            conn.execute("""
                UPDATE manuscripts
                SET verification_status = ?, updated_at = CURRENT_TIMESTAMP
                WHERE manuscript_id = ?
            """, (status, manuscript_id))
            conn.commit()

    updated = manuscript_row(manuscript_id)
    return {"ok": True, "message": f"Verification status set to {status}", "manuscript": public_record(updated)}


@app.get("/api/verification/queue")
def verification_queue(
    q: Optional[str] = None,
    page: int = Query(default=1, ge=1),
    page_size: int = Query(default=24, ge=1, le=100),
):
    # Preprocessed-only images are not reviewable yet. Queue only records that contain OCR text.
    where = [
        "LOWER(COALESCE(verification_status, '')) = 'pending'",
        "(COALESCE(TRIM(raw_ocr_text), '') <> '' OR COALESCE(TRIM(cleaned_text), '') <> '' OR COALESCE(TRIM(verified_text), '') <> '')",
    ]
    params: list[Any] = []
    if q and q.strip():
        wildcard = f"%{q.strip()}%"
        where.append("(manuscript_id LIKE ? OR title LIKE ? OR original_filename LIKE ? OR raw_ocr_text LIKE ? OR cleaned_text LIKE ?)")
        params.extend([wildcard] * 5)
    where_sql = " AND ".join(where)
    offset = (page - 1) * page_size
    with db_connection() as conn:
        total = conn.execute(f"SELECT COUNT(*) FROM manuscripts WHERE {where_sql}", params).fetchone()[0]
        rows = conn.execute(
            MANUSCRIPT_SELECT + f" WHERE {where_sql} ORDER BY COALESCE(manuscripts.updated_at, manuscripts.created_at) DESC LIMIT ? OFFSET ?",
            [*params, page_size, offset],
        ).fetchall()
    return {
        "items": [public_record(row) for row in rows],
        "total": int(total),
        "page": page,
        "pageSize": page_size,
        "totalPages": (int(total) + page_size - 1) // page_size,
    }


@app.get("/api/verification/summary")
def verification_summary():
    stats = dashboard_payload()
    return {
        "pendingReview": stats["pendingReview"],
        "verified": stats["verified"],
        "pendingVerification": stats["pendingVerification"],
    }



# -----------------------------------------------------------------------------
# TRANSLATION: IndicTrans2 distilled inference checkpoints
# -----------------------------------------------------------------------------

SUPPORTED_TRANSLATION_LANGUAGES: dict[str, str] = {
    # IndicTrans2 language/script inventory published by AI4Bharat.
    "asm_Beng": "Assamese",
    "ben_Beng": "Bengali",
    "brx_Deva": "Bodo",
    "doi_Deva": "Dogri",
    "eng_Latn": "English",
    "gom_Deva": "Konkani",
    "guj_Gujr": "Gujarati",
    "hin_Deva": "Hindi",
    "kan_Knda": "Kannada",
    "kas_Arab": "Kashmiri (Arabic script)",
    "kas_Deva": "Kashmiri (Devanagari)",
    "mai_Deva": "Maithili",
    "mal_Mlym": "Malayalam",
    "mar_Deva": "Marathi",
    "mni_Beng": "Manipuri (Bengali script)",
    "mni_Mtei": "Manipuri (Meitei script)",
    "npi_Deva": "Nepali",
    "ory_Orya": "Odia",
    "pan_Guru": "Punjabi",
    "san_Deva": "Sanskrit",
    "sat_Olck": "Santali",
    "snd_Arab": "Sindhi (Arabic script)",
    "snd_Deva": "Sindhi (Devanagari)",
    "tam_Taml": "Tamil",
    "tel_Telu": "Telugu",
    "urd_Arab": "Urdu",
}

TRANSLATION_MODEL_IDS = {
    "indic_to_english": "ai4bharat/indictrans2-indic-en-dist-200M",
    "english_to_indic": "ai4bharat/indictrans2-en-indic-dist-200M",
    "indic_to_indic": "ai4bharat/indictrans2-indic-indic-dist-320M",
}
TRANSLATION_JOBS: dict[str, dict[str, Any]] = {}
TRANSLATION_JOBS_LOCK = threading.Lock()
_TRANSLATION_MODEL_CACHE: dict[str, Any] = {}
_TRANSLATION_MODEL_LOCK = threading.Lock()
_TRANSLATION_IMPORT_ERROR = ""
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Current IndicTransToolkit import, followed by the package's documented fallback.
    try:
        from IndicTransToolkit import IndicProcessor
    except ImportError:
        try:
            from IndicTransToolkit.IndicTransToolkit import IndicProcessor
        except ImportError:
            from IndicTransToolkit.processor import IndicProcessor

    _TRANSLATION_IMPORT_ERROR = ""
except Exception as _translation_import_exc:
    _TRANSLATION_IMPORT_ERROR = f"{type(_translation_import_exc).__name__}: {_translation_import_exc}"
    print("Translation dependency import failed:", _TRANSLATION_IMPORT_ERROR)


def _translation_model_key(source_language: str, target_language: str) -> str:
    if source_language == "eng_Latn" and target_language != "eng_Latn":
        return "english_to_indic"
    if target_language == "eng_Latn":
        return "indic_to_english"
    return "indic_to_indic"


def _get_translation_model(source_language: str, target_language: str):
    """Load and cache the appropriate authenticated IndicTrans2 checkpoint."""
    if _TRANSLATION_IMPORT_ERROR:
        raise RuntimeError(
            "Translation dependencies are not ready. Run the dependency-install cell, "
            "then rerun this backend cell. " + _TRANSLATION_IMPORT_ERROR
        )

    model_key = _translation_model_key(source_language, target_language)

    # One load per direction; also prevents duplicate downloads on concurrent requests.
    with _TRANSLATION_MODEL_LOCK:
        cached = _TRANSLATION_MODEL_CACHE.get(model_key)
        if cached is not None:
            return cached

        model_name = TRANSLATION_MODEL_IDS[model_key]
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Prefer the explicitly supplied runtime token; fall back to huggingface-cli login.
        from huggingface_hub import get_token
        hf_token = (os.environ.get("HF_TOKEN", "") or get_token() or "").strip()
        if not hf_token:
            raise RuntimeError(
                "Hugging Face authentication is missing. Run the notebook's HF authentication cell "
                "before starting/restarting the API."
            )

        cache_dir = str(MODEL_CACHE_DIR)
        print(f"Loading IndicTrans2 checkpoint: {model_name}")
        print(f"Device: {device}; model cache: {cache_dir}")

        load_kwargs = {
            "trust_remote_code": True,
            "token": hf_token,
            "cache_dir": cache_dir,
        }
        if device == "cuda":
            load_kwargs["torch_dtype"] = torch.float16

        # Pass the token to BOTH calls: custom tokenizer/config/model code may be gated too.
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            token=hf_token,
            cache_dir=cache_dir,
        )
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            **load_kwargs,
        )
        model.to(device)
        model.eval()

        processor = IndicProcessor(inference=True)
        cached = (tokenizer, model, processor, device)
        _TRANSLATION_MODEL_CACHE[model_key] = cached

        print(f"Successfully loaded and cached: {model_name}")
        return cached


def _translation_segments(text: str, max_chars: int = 420) -> list[tuple[str, str]]:
    """Split text into manageable sentence chunks while retaining paragraph breaks."""
    segments: list[tuple[str, str]] = []
    for line in text.splitlines() or [text]:
        if not line.strip():
            segments.append(("blank", ""))
            continue
        parts = re.split(r"(?<=[।॥.!?])\s+", line.strip())
        for part in parts:
            if not part:
                continue
            if len(part) <= max_chars:
                segments.append(("text", part))
                continue
            words = part.split()
            current = ""
            for word in words:
                candidate = f"{current} {word}".strip()
                if current and len(candidate) > max_chars:
                    segments.append(("text", current))
                    current = word
                else:
                    current = candidate
            if current:
                segments.append(("text", current))
    return segments


def _translate_text_indictrans2(text: str, source_language: str, target_language: str) -> str:
    if source_language not in SUPPORTED_TRANSLATION_LANGUAGES:
        raise ValueError(f"Unsupported source language code: {source_language}")
    if target_language not in SUPPORTED_TRANSLATION_LANGUAGES:
        raise ValueError(f"Unsupported target language code: {target_language}")
    if source_language == target_language:
        raise ValueError("Source and target languages must be different.")
    if not clean_text(text):
        raise ValueError("There is no OCR text to translate. Verify or clean the OCR text first.")

    tokenizer, model, processor, device = _get_translation_model(source_language, target_language)
    segments = _translation_segments(text)
    text_segments = [segment for kind, segment in segments if kind == "text"]
    translated_segments: list[str] = []
    batch_size = 4
    for start in range(0, len(text_segments), batch_size):
        batch_sentences = text_segments[start:start + batch_size]
        preprocessed = processor.preprocess_batch(
            batch_sentences, src_lang=source_language, tgt_lang=target_language
        )
        inputs = tokenizer(
            preprocessed,
            truncation=True,
            padding="longest",
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
        ).to(device)
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=4,
                num_return_sequences=1,
            )
        decoded = tokenizer.batch_decode(
            generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        translated_segments.extend(processor.postprocess_batch(decoded, lang=target_language))

    output_lines: list[str] = []
    cursor = 0
    current_line: list[str] = []
    for kind, _segment in segments:
        if kind == "blank":
            output_lines.append(" ".join(current_line).strip())
            current_line = []
        else:
            current_line.append(translated_segments[cursor])
            cursor += 1
    if current_line or not output_lines:
        output_lines.append(" ".join(current_line).strip())
    return "\n".join(output_lines).strip()


def _manuscript_source(manuscript_id: str) -> tuple[str, str]:
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found.")
    source_text = (
        clean_text(row["verified_text"])
        or clean_text(row["cleaned_text"])
        or clean_text(row["raw_ocr_text"])
    )
    if not source_text:
        raise HTTPException(status_code=409, detail="This manuscript has no OCR text to translate yet.")
    return source_text, hashlib.sha256(source_text.encode("utf-8")).hexdigest()


def _save_translation(manuscript_id: str, source_language: str, target_language: str, source_text: str, translated_text: str, review_status: str = "machine_generated") -> dict[str, Any]:
    if target_language not in SUPPORTED_TRANSLATION_LANGUAGES:
        raise ValueError(f"Unsupported target language: {target_language}")
    source_hash = hashlib.sha256(source_text.encode("utf-8")).hexdigest()
    language_name = SUPPORTED_TRANSLATION_LANGUAGES[target_language]
    with db_connection() as conn:
        conn.execute("""
            INSERT INTO manuscript_translations (
                manuscript_id, language_code, language_name, source_language,
                source_hash, source_text, translated_text, engine, review_status,
                created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 'IndicTrans2', ?, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
            ON CONFLICT(manuscript_id, language_code) DO UPDATE SET
                language_name=excluded.language_name,
                source_language=excluded.source_language,
                source_hash=excluded.source_hash,
                source_text=excluded.source_text,
                translated_text=excluded.translated_text,
                engine=excluded.engine,
                review_status=excluded.review_status,
                updated_at=CURRENT_TIMESTAMP
        """, (manuscript_id, target_language, language_name, source_language,
              source_hash, source_text, translated_text, review_status))
        conn.commit()
    return {
        "manuscriptId": manuscript_id,
        "languageCode": target_language,
        "languageName": language_name,
        "sourceLanguage": source_language,
        "translatedText": translated_text,
        "engine": "IndicTrans2",
        "reviewStatus": review_status,
        "sourceHash": source_hash,
        "isCurrent": True,
    }


def _translate_and_store(manuscript_id: str, source_language: str, target_language: str) -> dict[str, Any]:
    source_text, _source_hash = _manuscript_source(manuscript_id)
    translated_text = _translate_text_indictrans2(source_text, source_language, target_language)
    return _save_translation(
        manuscript_id, source_language, target_language, source_text, translated_text
    )


def _translation_job_update(job_id: str, **changes):
    with TRANSLATION_JOBS_LOCK:
        if job_id in TRANSLATION_JOBS:
            TRANSLATION_JOBS[job_id].update(changes)
            TRANSLATION_JOBS[job_id]["updatedAt"] = datetime.now(timezone.utc).isoformat()


def _translation_job_snapshot(job_id: str) -> Optional[dict[str, Any]]:
    with TRANSLATION_JOBS_LOCK:
        item = TRANSLATION_JOBS.get(job_id)
        return copy.deepcopy(item) if item else None


def _run_translation_job(job_id: str, manuscript_id: str, source_language: str, target_languages: list[str]):
    results: list[dict[str, Any]] = []
    errors: list[str] = []
    try:
        for index, target_language in enumerate(target_languages, start=1):
            language_name = SUPPORTED_TRANSLATION_LANGUAGES[target_language]
            _translation_job_update(
                job_id,
                status="processing",
                currentLanguage=language_name,
                currentStage=f"Translating to {language_name}",
                progress=int((index - 1) * 100 / max(len(target_languages), 1)),
            )
            try:
                result = _translate_and_store(manuscript_id, source_language, target_language)
                result["translatedTextPreview"] = result["translatedText"][:500]
                results.append(result)
            except Exception as exc:
                errors.append(f"{language_name}: {exc}")
            _translation_job_update(
                job_id,
                completedLanguages=index,
                results=copy.deepcopy(results),
                errors=list(errors),
                progress=int(index * 100 / max(len(target_languages), 1)),
            )
        final_status = "completed" if not errors else ("completed_with_errors" if results else "failed")
        _translation_job_update(
            job_id, status=final_status,
            currentStage="Translations saved" if results else "Translation failed",
            progress=100 if results else 0,
            results=results, errors=errors,
        )
    except Exception as exc:
        _translation_job_update(job_id, status="failed", currentStage="Translation failed", errors=errors + [str(exc)], progress=0)


# -----------------------------------------------------------------------------
# MODULE 1 + 2 OCR PIPELINE: same functions and OCR configuration as the user's
# uploaded combined notebook. Tesseract uses its pretrained `san` and
# `Devanagari` language/script data; there are no custom project-trained weights.
# -----------------------------------------------------------------------------

import cv2
import numpy as np
import pytesseract
from PIL import Image
import fitz  # PyMuPDF: render pages from PDF bundles without poppler
from datetime import datetime

MAX_FILE_BYTES = 50 * 1024 * 1024
MAX_JOB_BYTES = 100 * 1024 * 1024
MAX_PAGES_PER_JOB = 100
ALLOWED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".bmp"}
ALLOWED_UPLOAD_EXTENSIONS = ALLOWED_IMAGE_EXTENSIONS | {".pdf", ".zip"}
PROCESSING_JOBS: dict[str, dict[str, Any]] = {}
PROCESSING_JOBS_LOCK = threading.Lock()


def _available_tesseract_models() -> set[str]:
    result = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True, check=True)
    return set(result.stdout.splitlines()[1:])


def _is_within(candidate: Path, root: Path) -> bool:
    try:
        candidate.relative_to(root)
        return True
    except ValueError:
        return False


def _safe_filename(value: str) -> str:
    # Do not trust a client-supplied path; retain only a safe filename.
    name = re.split(r"[\\/]", value or "upload.bin")[-1]
    name = re.sub(r"[^A-Za-z0-9._() -]+", "_", name).strip(" .")
    return (name or "upload.bin")[:160]


def _update_job(job_id: str, **changes):
    with PROCESSING_JOBS_LOCK:
        if job_id in PROCESSING_JOBS:
            PROCESSING_JOBS[job_id].update(changes)
            PROCESSING_JOBS[job_id]["updatedAt"] = datetime.now(timezone.utc).isoformat()


def _job_snapshot(job_id: str) -> Optional[dict[str, Any]]:
    with PROCESSING_JOBS_LOCK:
        job = PROCESSING_JOBS.get(job_id)
        return copy.deepcopy(job) if job else None


def _correct_sanskrit_ocr_text(raw_text: str) -> dict[str, Any]:
    """Port of correct_sanskrit_ocr_text() from the uploaded Module 2 notebook."""
    if not raw_text:
        return {"corrected_text": "", "changes": []}

    changes = []
    text = raw_text
    normalized = unicodedata.normalize("NFC", text)
    normalized = re.sub(r"[\u200b-\u200f\ufeff]", "", normalized)
    if normalized != text:
        changes.append("Unicode NFC & zero-width character normalization")
        text = normalized

    cleaned_lines = re.sub(r"[_~─━═]{2,}", " ", text)
    if cleaned_lines != text:
        changes.append("Removed line/border noise characters")
        text = cleaned_lines

    stripped = re.sub(r"[^\u0900-\u097F\s.,!?;:'\"()\-–—]", " ", text)
    if stripped != text:
        changes.append("Removed non-Devanagari OCR noise tokens")
        text = stripped

    replacements = {"़़": "़"}
    for wrong, right in replacements.items():
        if wrong in text:
            text = text.replace(wrong, right)
            changes.append(f"Replaced '{wrong}' -> '{right}'")

    spaced = re.sub(r"\s*।\s*", " । ", text)
    spaced = re.sub(r"\s*॥\s*", " ॥ ", spaced)
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in spaced.splitlines()]
    final_text = "\n".join(line for line in lines if line).strip()
    if final_text != text:
        changes.append("Normalized punctuation spacing and whitespace")
    return {"corrected_text": final_text, "changes": changes}


def _render_pdf_bytes(pdf_bytes: bytes, output_dir: Path, base_name: str) -> list[dict[str, str]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    pages = []
    document = fitz.open(stream=pdf_bytes, filetype="pdf")
    try:
        if document.page_count > MAX_PAGES_PER_JOB:
            raise ValueError(f"PDF has {document.page_count} pages; limit is {MAX_PAGES_PER_JOB} per job.")
        for page_index in range(document.page_count):
            page = document.load_page(page_index)
            pixmap = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
            target = output_dir / f"{base_name}_page_{page_index + 1:04d}.png"
            pixmap.save(str(target))
            pages.append({"path": str(target), "filename": f"{base_name} · Page {page_index + 1}"})
    finally:
        document.close()
    return pages


def _expand_input_file(input_path: Path, original_name: str, expanded_dir: Path) -> list[dict[str, str]]:
    suffix = Path(original_name).suffix.lower()
    expanded_dir.mkdir(parents=True, exist_ok=True)
    if suffix in ALLOWED_IMAGE_EXTENSIONS:
        return [{"path": str(input_path), "filename": original_name}]
    if suffix == ".pdf":
        return _render_pdf_bytes(input_path.read_bytes(), expanded_dir, Path(original_name).stem or "document")
    if suffix == ".zip":
        pages = []
        with zipfile.ZipFile(input_path, "r") as archive:
            members = [info for info in archive.infolist() if not info.is_dir()]
            members.sort(key=lambda item: item.filename.lower())
            for member_index, info in enumerate(members):
                inner_name = _safe_filename(info.filename)
                inner_suffix = Path(inner_name).suffix.lower()
                if inner_suffix not in ALLOWED_IMAGE_EXTENSIONS and inner_suffix != ".pdf":
                    continue
                if info.file_size > MAX_FILE_BYTES:
                    raise ValueError(f"Bundle item {inner_name} exceeds the 50 MB file limit.")
                payload = archive.read(info)
                if inner_suffix == ".pdf":
                    pages.extend(_render_pdf_bytes(
                        payload, expanded_dir / f"bundle_{member_index:03d}", Path(inner_name).stem or "document"
                    ))
                else:
                    destination = expanded_dir / f"bundle_{member_index:04d}_{inner_name}"
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    destination.write_bytes(payload)
                    pages.append({"path": str(destination), "filename": inner_name})
                if len(pages) > MAX_PAGES_PER_JOB:
                    raise ValueError(f"Bundle exceeds the {MAX_PAGES_PER_JOB}-page limit.")
        return pages
    raise ValueError(f"Unsupported file type: {suffix or '(no extension)'}")


def _process_one_page(job_id: str, page_number: int, total_pages: int, page: dict[str, str], page_root: Path) -> dict[str, Any]:
    """Run Module 1 preprocessing, Module 2 OCR, and Module 2 text correction."""
    source_path = Path(page["path"]).resolve()
    filename = page["filename"]
    page_dir = page_root / f"page_{page_number:04d}"
    page_dir.mkdir(parents=True, exist_ok=True)

    stage_offsets = {
        "Preprocessing image": 0.12,
        "Running Sanskrit/Devanagari OCR": 0.38,
        "Normalizing OCR text": 0.78,
        "Saving manuscript record": 0.92,
    }

    def report(stage: str):
        base = 6 + 90 * ((page_number - 1 + stage_offsets[stage]) / max(total_pages, 1))
        _update_job(
            job_id,
            status="processing",
            currentStage=stage,
            currentFile=filename,
            currentPage=page_number,
            progress=max(1, min(99, int(base))),
        )

    report("Preprocessing image")
    image = cv2.imread(str(source_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("The selected page could not be decoded as an image.")

    # Module 1's dataset preprocessing: grayscale -> Gaussian blur -> CLAHE -> adaptive threshold.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    denoised_module1 = cv2.GaussianBlur(gray, (3, 3), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised_module1)
    binary = cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10
    )

    original_extension = source_path.suffix.lower()
    if original_extension not in ALLOWED_IMAGE_EXTENSIONS:
        original_extension = ".png"
    original_copy = page_dir / f"original{original_extension}"
    if source_path != original_copy:
        shutil.copy2(source_path, original_copy)
    enhanced_path = page_dir / "enhanced.png"
    binary_path = page_dir / "binary.png"
    if not cv2.imwrite(str(enhanced_path), enhanced):
        raise RuntimeError("Could not save the enhanced grayscale image.")
    if not cv2.imwrite(str(binary_path), binary):
        raise RuntimeError("Could not save the binary image.")

    # Module 2's OCR-specific cleaner: grayscale -> Fast Non-Local Means denoising.
    report("Running Sanskrit/Devanagari OCR")
    ocr_denoised = cv2.fastNlMeansDenoising(gray, h=10)
    cleaned_pil = Image.fromarray(ocr_denoised)
    raw_text = pytesseract.image_to_string(
        cleaned_pil, lang="san+Devanagari", config="--psm 6"
    )

    report("Normalizing OCR text")
    correction_result = _correct_sanskrit_ocr_text(raw_text)
    corrected_text = correction_result["corrected_text"]

    report("Saving manuscript record")
    manuscript_id = f"UPL-{job_id[:8].upper()}-{page_number:04d}"
    now_title = f"Uploaded Manuscript — {Path(filename).stem[:100]}"
    description = (
        "Uploaded manuscript page processed using the Module 1 image-preprocessing steps and "
        "the Module 2 Tesseract OCR profile (san+Devanagari, --psm 6). "
        "Sanskrit / Devanagari are the selected OCR profile, not an automatic script-identification result."
    )
    with db_connection() as conn:
        conn.execute("""
            INSERT INTO manuscripts (
                manuscript_id, title, original_filename, original_path,
                enhanced_path, binary_path, raw_ocr_text, cleaned_text,
                translated_text, verified_text, language, script, category,
                description, source, processing_status, verification_status,
                created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL, NULL, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
        """, (
            manuscript_id, now_title, filename, str(original_copy),
            str(enhanced_path), str(binary_path), raw_text, corrected_text,
            "Sanskrit", "Devanagari", "Website Upload", description,
            "BharatLipi Website", "OCR Completed", "Pending",
        ))
        conn.commit()

    return {
        "pageNumber": page_number,
        "filename": filename,
        "status": "completed",
        "manuscriptId": manuscript_id,
        "characterCount": len(corrected_text),
        "cleanedTextPreview": corrected_text[:500],
        "correctionSteps": correction_result["changes"],
    }


def _run_processing_job(job_id: str, uploaded_files: list[dict[str, str]], translation_languages: list[str], source_language: str):
    job_root = UPLOADS_DIR / job_id
    expanded_dir = job_root / "expanded"
    page_root = job_root / "processed"
    all_pages = []
    errors = []
    try:
        _update_job(job_id, status="processing", currentStage="Extracting pages from bundle", progress=3)
        for item in uploaded_files:
            all_pages.extend(_expand_input_file(Path(item["path"]), item["filename"], expanded_dir / Path(item["filename"]).stem))
            if len(all_pages) > MAX_PAGES_PER_JOB:
                raise ValueError(f"Upload exceeds the {MAX_PAGES_PER_JOB}-page limit.")
        if not all_pages:
            raise ValueError("No supported manuscript images or PDF pages were found in the upload.")

        _update_job(
            job_id,
            status="processing",
            currentStage="Preparing page image",
            totalPages=len(all_pages),
            currentPage=0,
            currentFile="",
            progress=6,
        )
        results = []
        translation_errors = []
        for page_number, page in enumerate(all_pages, start=1):
            try:
                result = _process_one_page(job_id, page_number, len(all_pages), page, page_root)
                result["translations"] = []
                result["translationErrors"] = []
                if translation_languages:
                    for language_index, target_language in enumerate(translation_languages, start=1):
                        language_name = SUPPORTED_TRANSLATION_LANGUAGES[target_language]
                        progress = int(6 + 92 * ((page_number - 1) + (0.90 + 0.09 * language_index / len(translation_languages))) / max(len(all_pages), 1))
                        _update_job(
                            job_id, status="processing",
                            currentStage=f"Translating page {page_number} to {language_name}",
                            currentFile=page.get("filename", ""), currentPage=page_number,
                            progress=max(1, min(99, progress)),
                        )
                        try:
                            saved_translation = _translate_and_store(
                                result["manuscriptId"], source_language, target_language
                            )
                            result["translations"].append({
                                "languageCode": target_language,
                                "languageName": language_name,
                                "characterCount": len(saved_translation["translatedText"]),
                                "translatedTextPreview": saved_translation["translatedText"][:320],
                                "reviewStatus": saved_translation["reviewStatus"],
                            })
                        except Exception as translation_error:
                            warning = f"Page {page_number} ({page.get('filename', 'unknown')}) → {language_name}: {translation_error}"
                            result["translationErrors"].append(warning)
                            translation_errors.append(warning)
                results.append(result)
            except Exception as page_error:
                message = f"Page {page_number} ({page.get('filename', 'unknown')}): {page_error}"
                errors.append(message)
                results.append({
                    "pageNumber": page_number,
                    "filename": page.get("filename", "unknown"),
                    "status": "failed",
                    "error": str(page_error),
                })
            _update_job(
                job_id,
                completedPages=page_number,
                results=copy.deepcopy(results),
                errors=list(errors),
                translationErrors=list(translation_errors),
                progress=max(6, min(99, int(6 + 92 * page_number / max(len(all_pages), 1)))),
            )

        successful = sum(1 for result in results if result.get("status") == "completed")
        final_status = "completed" if not errors and not translation_errors else ("completed_with_errors" if successful else "failed")
        _update_job(
            job_id,
            status=final_status,
            currentStage=("Completed with translation warnings" if translation_errors and successful else ("Completed" if successful else "Processing failed")),
            currentPage=len(all_pages),
            completedPages=successful,
            currentFile="",
            progress=100 if successful else 0,
            results=results,
            errors=errors,
            translationErrors=translation_errors,
        )
    except Exception as exc:
        _update_job(
            job_id,
            status="failed",
            currentStage="Processing failed",
            errors=errors + [str(exc)],
            progress=0,
        )


@app.post("/api/processing/jobs")
async def create_processing_job(
    background_tasks: BackgroundTasks,
    files: list[UploadFile] = File(...),
    translation_languages: list[str] = Form(default=[]),
    source_language: str = Form(default="san_Deva"),
):
    """Accept images/PDF/ZIP, run OCR, then optionally translate each page."""
    if not files:
        raise HTTPException(status_code=422, detail="Select at least one file.")
    if len(files) > 30:
        raise HTTPException(status_code=413, detail="Upload at most 30 files in one job.")
    translation_languages = list(dict.fromkeys(
        language for language in translation_languages
        if language in SUPPORTED_TRANSLATION_LANGUAGES and language != source_language
    ))
    if source_language not in SUPPORTED_TRANSLATION_LANGUAGES:
        raise HTTPException(status_code=422, detail="Unsupported source language.")

    job_id = uuid.uuid4().hex[:12]
    input_dir = UPLOADS_DIR / job_id / "inputs"
    input_dir.mkdir(parents=True, exist_ok=True)
    saved_files = []
    total_bytes = 0
    try:
        for index, upload in enumerate(files, start=1):
            original_name = _safe_filename(upload.filename or f"upload_{index}")
            suffix = Path(original_name).suffix.lower()
            if suffix not in ALLOWED_UPLOAD_EXTENSIONS:
                raise HTTPException(status_code=415, detail=f"Unsupported file type for {original_name}.")
            destination = input_dir / f"{index:03d}_{original_name}"
            file_bytes = 0
            with destination.open("wb") as output:
                while True:
                    chunk = await upload.read(1024 * 1024)
                    if not chunk:
                        break
                    file_bytes += len(chunk)
                    total_bytes += len(chunk)
                    if file_bytes > MAX_FILE_BYTES:
                        raise HTTPException(status_code=413, detail=f"{original_name} exceeds the 50 MB per-file limit.")
                    if total_bytes > MAX_JOB_BYTES:
                        raise HTTPException(status_code=413, detail="Combined upload exceeds the 100 MB limit.")
                    output.write(chunk)
            saved_files.append({"path": str(destination), "filename": original_name})
    except Exception:
        shutil.rmtree(UPLOADS_DIR / job_id, ignore_errors=True)
        raise
    finally:
        for upload in files:
            await upload.close()

    now = datetime.now(timezone.utc).isoformat()
    job = {
        "jobId": job_id,
        "status": "queued",
        "progress": 0,
        "currentStage": "Queued for OCR processing",
        "currentFile": "",
        "currentPage": 0,
        "totalPages": 0,
        "completedPages": 0,
        "totalFiles": len(saved_files),
        "results": [],
        "errors": [],
        "translationLanguages": translation_languages,
        "translationLanguageNames": [SUPPORTED_TRANSLATION_LANGUAGES[code] for code in translation_languages],
        "translationErrors": [],
        "sourceLanguage": source_language,
        "createdAt": now,
        "updatedAt": now,
    }
    with PROCESSING_JOBS_LOCK:
        PROCESSING_JOBS[job_id] = job
    background_tasks.add_task(_run_processing_job, job_id, saved_files, translation_languages, source_language)
    return copy.deepcopy(job)


@app.get("/api/processing/jobs/{job_id}")
def get_processing_job(job_id: str):
    job = _job_snapshot(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="Processing job not found. The Colab runtime may have restarted.")
    return job


@app.get("/api/ocr/status")
def get_ocr_status():
    try:
        langs = sorted(_available_tesseract_models())
        return {
            "ready": {"san", "Devanagari"}.issubset(set(langs)),
            "engine": "Tesseract",
            "languageProfile": "san+Devanagari",
            "pageSegmentationMode": 6,
            "availableLanguages": langs,
        }
    except Exception as exc:
        return {"ready": False, "engine": "Tesseract", "error": str(exc)}


# -----------------------------------------------------------------------------
# TRANSLATION API
# -----------------------------------------------------------------------------

@app.get("/api/translation/languages")
def get_translation_languages():
    return {
        "engine": "IndicTrans2",
        "languages": [
            {"code": code, "name": name} for code, name in SUPPORTED_TRANSLATION_LANGUAGES.items()
        ],
        "translationImportReady": not bool(_TRANSLATION_IMPORT_ERROR),
        "translationImportError": _TRANSLATION_IMPORT_ERROR or None,
    }


@app.get("/api/translation/status")
def get_translation_status():
    try:
        from huggingface_hub import get_token
        auth_configured = bool((os.environ.get("HF_TOKEN", "") or get_token() or "").strip())
    except Exception:
        auth_configured = bool(os.environ.get("HF_TOKEN"))
    dependencies_ready = not bool(_TRANSLATION_IMPORT_ERROR)
    return {
        "ready": dependencies_ready and auth_configured,
        "dependenciesReady": dependencies_ready,
        "authenticationConfigured": auth_configured,
        "engine": "IndicTrans2",
        "modelLoaded": bool(_TRANSLATION_MODEL_CACHE),
        "device": "cuda" if dependencies_ready and torch.cuda.is_available() else "cpu",
        "languageCount": len(SUPPORTED_TRANSLATION_LANGUAGES),
        "models": TRANSLATION_MODEL_IDS,
        "error": _TRANSLATION_IMPORT_ERROR or (None if auth_configured else "Hugging Face token not configured"),
    }


@app.get("/api/manuscripts/{manuscript_id}/translations")
def get_saved_translations(manuscript_id: str):
    row = manuscript_row(manuscript_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Manuscript not found.")
    source_text = (
        clean_text(row["verified_text"])
        or clean_text(row["cleaned_text"])
        or clean_text(row["raw_ocr_text"])
    )
    source_hash = hashlib.sha256(source_text.encode("utf-8")).hexdigest()
    with db_connection() as conn:
        rows = conn.execute("""
            SELECT manuscript_id, language_code, language_name, source_language,
                   source_hash, translated_text, engine, review_status, created_at, updated_at
            FROM manuscript_translations
            WHERE manuscript_id = ?
            ORDER BY language_name
        """, (manuscript_id,)).fetchall()
    return {
        "manuscriptId": manuscript_id,
        "sourceCharacterCount": len(source_text),
        "items": [
            {
                "manuscriptId": row["manuscript_id"],
                "languageCode": row["language_code"],
                "languageName": row["language_name"],
                "sourceLanguage": row["source_language"],
                "translatedText": row["translated_text"],
                "engine": row["engine"],
                "reviewStatus": row["review_status"],
                "createdAt": row["created_at"],
                "updatedAt": row["updated_at"],
                "isCurrent": row["source_hash"] == source_hash,
            } for row in rows
        ],
    }


@app.post("/api/manuscripts/{manuscript_id}/translations/jobs")
def create_translation_job(manuscript_id: str, background_tasks: BackgroundTasks, payload: dict[str, Any] = Body(...)):
    _source_text, _source_hash = _manuscript_source(manuscript_id)
    source_language = clean_text(payload.get("sourceLanguage") or "san_Deva")
    target_languages = payload.get("targetLanguages") or []
    if not isinstance(target_languages, list) or not target_languages:
        raise HTTPException(status_code=422, detail="Select at least one target language.")
    if source_language not in SUPPORTED_TRANSLATION_LANGUAGES:
        raise HTTPException(status_code=422, detail="Unsupported source language.")
    target_languages = list(dict.fromkeys(str(code) for code in target_languages))
    unsupported = [code for code in target_languages if code not in SUPPORTED_TRANSLATION_LANGUAGES]
    if unsupported:
        raise HTTPException(status_code=422, detail="Unsupported target language code(s): " + ", ".join(unsupported))
    if source_language in target_languages:
        raise HTTPException(status_code=422, detail="A target language cannot be the same as the source language.")
    if len(target_languages) > 25:
        raise HTTPException(status_code=422, detail="Select no more than 25 target languages in one run.")

    job_id = uuid.uuid4().hex[:12]
    now = datetime.now(timezone.utc).isoformat()
    job = {
        "jobId": job_id, "manuscriptId": manuscript_id, "status": "queued",
        "progress": 0, "currentStage": "Queued for translation", "currentLanguage": "",
        "completedLanguages": 0, "totalLanguages": len(target_languages),
        "targetLanguages": target_languages,
        "targetLanguageNames": [SUPPORTED_TRANSLATION_LANGUAGES[code] for code in target_languages],
        "results": [], "errors": [], "createdAt": now, "updatedAt": now,
    }
    with TRANSLATION_JOBS_LOCK:
        TRANSLATION_JOBS[job_id] = job
    background_tasks.add_task(_run_translation_job, job_id, manuscript_id, source_language, target_languages)
    return copy.deepcopy(job)


@app.get("/api/translations/jobs/{job_id}")
def get_translation_job(job_id: str):
    job = _translation_job_snapshot(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="Translation job not found. The Colab runtime may have restarted.")
    return job


@app.put("/api/manuscripts/{manuscript_id}/translations/{language_code}")
def save_translation_edit(manuscript_id: str, language_code: str, payload: dict[str, Any] = Body(...)):
    source_text, _source_hash = _manuscript_source(manuscript_id)
    translated_text = clean_text(payload.get("translatedText"))
    if not translated_text:
        raise HTTPException(status_code=422, detail="Translation text cannot be empty.")
    with db_connection() as conn:
        existing = conn.execute(
            "SELECT source_language FROM manuscript_translations WHERE manuscript_id = ? AND language_code = ?",
            (manuscript_id, language_code),
        ).fetchone()
    if not existing:
        raise HTTPException(status_code=404, detail="Generate this language translation before editing it.")
    return _save_translation(
        manuscript_id, existing["source_language"], language_code,
        source_text, translated_text, review_status="human_edited",
    )


print("FastAPI application created.")
print("Available documentation route: /docs")


## 3B. Real translation smoke test (before starting the API)

This deliberately loads and runs both default Sanskrit translation directions used by the website: Sanskrit → English and Sanskrit → Hindi. It does not write test output into the manuscript database. The first run downloads the model weights into the Google Drive cache.

In [ ]:
# 3B. Run real IndicTrans2 inference tests for the website's default target languages

TEST_SOURCE_LANGUAGE = "san_Deva"
TEST_SOURCE_TEXT = "धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः। मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय॥"
TEST_TARGETS = [
    ("eng_Latn", "English"),
    ("hin_Deva", "Hindi"),
]

if _TRANSLATION_IMPORT_ERROR:
    raise RuntimeError(
        "Cannot run translation smoke tests because dependencies failed to import: "
        + _TRANSLATION_IMPORT_ERROR
    )

print("Translation device:", "GPU" if torch.cuda.is_available() else "CPU")
print("Source Sanskrit:", TEST_SOURCE_TEXT)

TRANSLATION_SMOKE_RESULTS = {}
for target_code, target_name in TEST_TARGETS:
    print(f"\nTesting Sanskrit → {target_name} ({target_code}) ...")
    translated = _translate_text_indictrans2(
        TEST_SOURCE_TEXT,
        TEST_SOURCE_LANGUAGE,
        target_code,
    )
    if not translated or not translated.strip():
        raise RuntimeError(f"IndicTrans2 returned empty output for {target_name}.")
    TRANSLATION_SMOKE_RESULTS[target_code] = translated
    print(f"{target_name} result:")
    print(translated)

print("\nSUCCESS: IndicTrans2 inference worked for both default translation targets.")
print("The two model checkpoints are now cached in memory for the API session.")


In [ ]:
# 4. Start the API inside this Colab runtime
import threading
import time
import requests
import uvicorn

API_HOST = "0.0.0.0"
API_PORT = 8000

# Avoid starting a second server if this cell is accidentally run again.
if "api_thread" in globals() and api_thread.is_alive():
    print(f"API server is already running on port {API_PORT}.")
else:
    api_config = uvicorn.Config(
        app,
        host=API_HOST,
        port=API_PORT,
        log_level="warning",
        access_log=True,
    )
    api_server = uvicorn.Server(api_config)
    api_thread = threading.Thread(target=api_server.run, daemon=True)
    api_thread.start()

for attempt in range(30):
    try:
        response = requests.get(f"http://127.0.0.1:{API_PORT}/api/health", timeout=2)
        if response.ok:
            print("Local API is running:", response.json())
            translation_status = requests.get(f"http://127.0.0.1:{API_PORT}/api/translation/status", timeout=10).json()
            print("Translation dependencies:", translation_status)
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI did not start. Check the cell output above for the server error.")


## 5. Create the public connection URL

The React website runs on your laptop, while this API runs inside Colab. A tunnel gives your laptop a temporary HTTPS address for the API.

This cell asks for your **ngrok authtoken** using a hidden password prompt. It is not printed or saved in the notebook. Do not paste the token into chat or into a code cell.


In [ ]:
# 5. Open an HTTPS tunnel to the Colab API

from getpass import getpass
from pyngrok import conf, ngrok
import requests

# Enter your ngrok authtoken
NGROK_AUTHTOKEN = getpass("Enter your ngrok authtoken: ").strip()

if not NGROK_AUTHTOKEN:
    raise ValueError("Ngrok authtoken is required.")

# Explicitly configure ngrok authentication
ngrok_config = conf.PyngrokConfig(
    auth_token=NGROK_AUTHTOKEN
)

# Close any previous ngrok process
try:
    ngrok.kill(pyngrok_config=ngrok_config)
except Exception:
    pass

# Verify that the local FastAPI server is running
local_health = requests.get(
    f"http://127.0.0.1:{API_PORT}/api/health",
    timeout=15
)

local_health.raise_for_status()

print("Local BharatLipi API is running!")

# Create a new HTTPS tunnel using the supplied token
tunnel = ngrok.connect(
    addr=API_PORT,
    proto="http",
    pyngrok_config=ngrok_config
)

PUBLIC_API_BASE_URL = tunnel.public_url.rstrip("/")

print("\nBharatLipi API is publicly accessible at:")
print(PUBLIC_API_BASE_URL)

# Verify public API access
health_response = requests.get(
    PUBLIC_API_BASE_URL + "/api/health",
    timeout=30
)

health_response.raise_for_status()

print("\nAPI health:", health_response.json())

print("\nSwagger documentation:")
print(PUBLIC_API_BASE_URL + "/docs")

print("\nAdd this to your frontend .env.local file:")
print(f"VITE_API_BASE_URL={PUBLIC_API_BASE_URL}")

## 6. Smoke tests

The earlier translation smoke-test cell has already loaded the English and Hindi checkpoints and run real model inference. This cell checks the public API, repository, OCR status, translation authentication/readiness, supported language list, and manuscript endpoints.


In [ ]:
# 6. Test core endpoints through the public URL
BASE_URL = PUBLIC_API_BASE_URL

checks = [
    ("Health", "/api/health"),
    ("Dashboard", "/api/dashboard"),
    ("Manuscript list", "/api/manuscripts?page=1&page_size=5"),
    ("Filter options", "/api/filters"),
    ("Verification queue", "/api/verification/queue?page=1&page_size=5"),
    ("OCR status", "/api/ocr/status"),
    ("Translation status", "/api/translation/status"),
    ("Translation languages", "/api/translation/languages"),
]

for label, route in checks:
    response = requests.get(BASE_URL + route, timeout=30)
    print(f"{label:22} HTTP {response.status_code}")
    response.raise_for_status()

health_data = requests.get(BASE_URL + "/api/health", timeout=20).json()
list_data = requests.get(BASE_URL + "/api/manuscripts?page=1&page_size=2", timeout=20).json()

print("\nHealth payload:", health_data)
print("Records returned:", len(list_data.get("items", [])), "of", list_data.get("total", 0))
if list_data.get("items"):
    sample_id = list_data["items"][0]["id"]
    detail = requests.get(f"{BASE_URL}/api/manuscripts/{sample_id}", timeout=20)
    detail.raise_for_status()
    print("Sample manuscript detail:", detail.json()["id"], detail.json()["originalFilename"])
    for image_type in ("enhanced", "binary"):
        image_response = requests.get(
            f"{BASE_URL}/api/manuscripts/{sample_id}/images/{image_type}",
            timeout=30,
        )
        print(f"{image_type.title()} image:", image_response.status_code,
              image_response.headers.get("content-type"),
              len(image_response.content), "bytes")

print("\nSmoke tests completed.")


## API contract for the React website

| Method | Route | Purpose |
|---|---|---|
| `GET` | `/api/health` | API/database/OCR/translation readiness |
| `GET` | `/api/dashboard` | Live repository statistics and recent manuscripts |
| `GET` | `/api/manuscripts` | Filtered and paginated repository |
| `GET` | `/api/manuscripts/{id}` | Manuscript details |
| `GET` | `/api/manuscripts/{id}/images/{enhanced|binary}` | Processed image variants |
| `GET` | `/api/manuscripts/{id}/text` | OCR text |
| `PUT` | `/api/manuscripts/{id}/text` | Save corrected OCR text |
| `PUT` | `/api/manuscripts/{id}/verification` | Save verification status |
| `GET` | `/api/verification/queue` | Pending manuscripts for review |
| `GET` | `/api/translation/languages` | Supported IndicTrans2 language codes |
| `GET` | `/api/translation/status` | Translation dependency/model status |
| `POST` | `/api/processing/jobs` | Upload images/PDF/ZIP, run OCR, and optionally translate every page |
| `GET` | `/api/processing/jobs/{job_id}` | Poll OCR + per-page translation progress |
| `GET` | `/api/manuscripts/{id}/translations` | Retrieve saved per-language translations |
| `POST` | `/api/manuscripts/{id}/translations/jobs` | Generate translations for an existing OCR record |
| `GET` | `/api/translations/jobs/{job_id}` | Poll translation progress |
| `PUT` | `/api/manuscripts/{id}/translations/{language_code}` | Save a human-edited translation |

### Translation behavior

- OCR source text is retained unchanged in the existing manuscript columns. Translations are stored in the additive `manuscript_translations` table with one row per manuscript/language.
- Translation records include their source-text hash. When OCR/verified text changes, the UI flags older translations as stale so they can be regenerated.
- Hindi and English are the default upload targets. The user can select any supported language/script option or run OCR only.
- Model checkpoints are loaded on first translation and cached in Google Drive. Run Colab with a GPU for much better speed.
- IndicTrans2 is a machine-translation model, not a scholarly guarantee of exact word-for-word equivalence. Sanskrit OCR and the generated translation must be reviewed by a subject expert before publication.
- Model download access is gated by Hugging Face. Accept the conditions on all three model pages above and sign in with a Hugging Face read token before starting the API.

### Storage and safety

The translation migration creates a new table using `CREATE TABLE IF NOT EXISTS`; existing manuscript rows, IDs, image paths, and the 1,199-record repository are not dropped or re-imported. Translation text is stored separately from OCR text.

Keep the Colab runtime and ngrok tunnel running while the local website is in use. After restarting Colab, re-run the notebook and update `VITE_API_BASE_URL` in the website `.env.local` with the new public URL.
